# Où part le temps ?

Ce notebook lit un JSON par jour dans `data/days/` (généré depuis `data/Planning.md`
par `planning_to_json.py`, un fichier par date pour que ça ne devienne pas
un seul gros fichier) et visualise la répartition du temps sur la période
choisie (voir `START_DAY` / `END_DAY` plus bas) :

1. **Frise horaire par jour** (façon Gantt) — tout ce qui a été fait, heure par heure.
2. **Total par catégorie** — où va le gros du temps.
3. **Heatmap jour × catégorie** — pour repérer les habitudes qui reviennent chaque jour.

La catégorisation (`CATEGORIES` ci-dessous) est une hypothèse de départ, à corriger
facilement si une activité est mal classée.

In [ ]:
import json
from pathlib import Path
from collections import defaultdict

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

LOCAL_FILE = Path("data/analysis.local.json")
LOCAL = json.loads(LOCAL_FILE.read_text(encoding="utf-8")) if LOCAL_FILE.exists() else {}

DAYS_DIR = Path("data/days")
data = {}
day_label = {}  # e.g. "AAAA-MM-JJ" -> "Jour JJ/MM"
for path in sorted(DAYS_DIR.glob("*.json")):
    day = json.loads(path.read_text(encoding="utf-8"))
    iso_date = day["date"]
    data[iso_date] = day
    dd, mm = iso_date.split("-")[2], iso_date.split("-")[1]
    day_label[iso_date] = f"{day['day_name']} {dd}/{mm}"

def to_hours(t):
    h, m = t.split(":")
    return int(h) + int(m) / 60

## Plage de dates à analyser

Chaque jour dans `data/days/` est nommé par sa vraie date ISO (`YYYY-MM-DD`),
déduite dans `planning_to_json.py` à partir d'une date d'ancrage pour le
premier jour du log + les numéros du mois pour chaque `# Nom jour`.

Laisse `START_DAY` / `END_DAY` à `None` pour ne pas tronquer de ce côté-là
(`None` pour les deux = toute la période disponible dans `data/days/`).

In [ ]:
START_DAY = LOCAL.get("start_day")  # ex: "AAAA-MM-JJ" -- inclus. None = depuis le tout premier jour dispo
END_DAY = LOCAL.get("end_day")  # ex: "AAAA-MM-JJ" -- inclus. None = jusqu'au tout dernier jour dispo

all_days = sorted(data.keys())

def _resolve(iso_date, default_idx):
    if iso_date is None:
        return default_idx
    if iso_date not in all_days:
        raise ValueError(f"{iso_date!r} n'est pas un jour connu. Jours disponibles: {all_days}")
    return all_days.index(iso_date)

start_idx = _resolve(START_DAY, 0)
end_idx = _resolve(END_DAY, len(all_days) - 1)
if start_idx > end_idx:
    raise ValueError(f"START_DAY ({START_DAY!r}) est après END_DAY ({END_DAY!r})")

days = all_days[start_idx:end_idx + 1]
print(f"Jours analysés ({len(days)}/{len(all_days)}):", [day_label[d] for d in days])

## Catégorisation

Mappe chaque libellé d'activité brut vers une catégorie plus large. Tout ce qui
n'est pas listé tombe dans `"Autre"` — la cellule suivante liste ces cas pour
que tu puisses les ranger si besoin. Tes catégories à toi vont dans `data/analysis.local.json`
(clé `categories`, avec `start_day` / `end_day` pour la plage de dates), qui reste local.

In [ ]:
# Réglages personnels (catégories, plage de dates) : lus dans analysis.local.json,
# un fichier local ignoré par git. Sans ce fichier, seules les catégories de base
# ci-dessous s'appliquent, et tout le reste tombe dans "Autre".
CATEGORIES = {
    "Work": "Travail",
    "Commute": "Trajet",
    "Walk": "Sport",
    "Scroll": "Écrans",
    "Video": "Écrans",
    "Nap": "Repos",
    "Social": "Social",
    "Cooking": "Corvées",
    "Chores": "Corvées",
    "Shower": "Corvées",
    "Study": "Travail",
    "Reading": "Travail",
}
CATEGORIES.update(LOCAL.get("categories", {}))

SLEEP_LABEL = "Sommeil"

CATEGORY_COLORS = {
    "Travail": "#4C72B0", "Sport": "#55A868", "Écrans": "#C44E52", "Repas": "#8172B2",
    "Repos": "#CCB974", "Trajet": "#64B5CD", "Social": "#B279A2", "Autre": "#A9A9A9",
    "Corvées": "#A0522D", SLEEP_LABEL: "#2E2E3A",
}
CATEGORY_COLORS.update(LOCAL.get("colors", {}))  # couleurs de tes catégories perso

uncategorized = sorted({
    a["activity"]
    for info in data.values()
    for a in info["activities"]
    if a["activity"] not in CATEGORIES
})
if uncategorized:
    print("Activités non catégorisées -> rangées dans 'Autre':")
    for a in uncategorized:
        print(" -", a)
else:
    print("Toutes les activités sont catégorisées.")

## 1. Frise horaire par jour

Chaque ligne est un jour ; l'axe horizontal est l'heure (0 = minuit du jour en
question). Le sommeil (entre l'heure de coucher notée et le réveil du
lendemain) est ajouté pour compléter la boucle de 24h, quand les deux
horaires sont connus.

In [ ]:
def build_timeline():
    rows = []  # (day, label, category, start_h, end_h)
    for i, day in enumerate(days):
        info = data[day]
        cursor = to_hours(info["wake_time"])
        for act in info["activities"]:
            dur = to_hours(act["end"]) - to_hours(act["start"])
            if dur < 0:
                dur += 24
            cat = CATEGORIES.get(act["activity"], "Autre")
            rows.append((day, act["activity"], cat, cursor, cursor + dur))
            cursor += dur

        # sleep block, bridging to next day's wake_time if both are known
        if info["sleep_time"] is not None and i + 1 < len(days):
            next_wake = data[days[i + 1]]["wake_time"]
            dur = to_hours(next_wake) - to_hours(info["sleep_time"])
            if dur <= 0:
                dur += 24
            rows.append((day, SLEEP_LABEL, SLEEP_LABEL, cursor, cursor + dur))

    return pd.DataFrame(rows, columns=["day", "activity", "category", "start_h", "end_h"])

timeline = build_timeline()
timeline.head()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

for i, day in enumerate(days):
    y = len(days) - i
    day_rows = timeline[timeline["day"] == day]
    for _, r in day_rows.iterrows():
        ax.barh(y, r["end_h"] - r["start_h"], left=r["start_h"], height=0.7,
                color=CATEGORY_COLORS.get(r["category"], "#999999"),
                edgecolor="white", linewidth=0.5)

ax.set_yticks(range(1, len(days) + 1))
ax.set_yticklabels([day_label[d] for d in reversed(days)])
max_h = timeline["end_h"].max()
ax.set_xticks(range(0, int(max_h) + 2, 2))
ax.set_xlabel("Heure")
ax.set_xlim(0, max_h + 0.5)
ax.set_title("Frise horaire de la semaine")

handles = [mpatches.Patch(color=c, label=cat) for cat, c in CATEGORY_COLORS.items()
           if cat in timeline["category"].unique()]
ax.legend(handles=handles, bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=9)
plt.tight_layout()
plt.show()

## 2. Total par catégorie sur la semaine

In [ ]:
cat_totals = (
    timeline.assign(hours=timeline["end_h"] - timeline["start_h"])
    .groupby("category")["hours"].sum()
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(cat_totals.index, cat_totals.values,
              color=[CATEGORY_COLORS.get(c, "#999999") for c in cat_totals.index])
ax.set_ylabel("Heures sur la semaine")
ax.set_title("Temps total par catégorie")
for b, v in zip(bars, cat_totals.values):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.3, f"{v:.1f}h", ha="center", fontsize=9)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

cat_totals.to_frame("hours").style.format("{:.1f}")

## 3. Heatmap jour × catégorie (heures)

In [ ]:
pivot = (
    timeline.assign(hours=timeline["end_h"] - timeline["start_h"])
    .pivot_table(index="day", columns="category", values="hours", aggfunc="sum", fill_value=0)
    .reindex(days)
)

fig, ax = plt.subplots(figsize=(10, 4.5))
im = ax.imshow(pivot.values, cmap="YlOrRd", aspect="auto")
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=30, ha="right")
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([day_label[d] for d in pivot.index])
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        v = pivot.values[i, j]
        if v > 0:
            ax.text(j, i, f"{v:.1f}", ha="center", va="center",
                    color="white" if v > pivot.values.max() / 2 else "black", fontsize=8)
fig.colorbar(im, ax=ax, label="heures")
ax.set_title("Heures par jour et par catégorie")
plt.tight_layout()
plt.show()

## Top activités brutes (sans regroupement)

Utile pour voir précisément quelles activités individuelles pèsent le plus,
sans le filtre des catégories.

In [ ]:
raw_totals = (
    timeline.assign(hours=timeline["end_h"] - timeline["start_h"])
    .groupby("activity")["hours"].sum()
    .sort_values(ascending=False)
    .head(15)
)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(raw_totals.index[::-1], raw_totals.values[::-1], color="#4C72B0")
ax.set_xlabel("Heures sur la semaine")
ax.set_title("Top 15 activités (brutes)")
plt.tight_layout()
plt.show()